# ViT from scratch on CIFAR-10 — paper implementation project (DL-065)

**Paper:** Dosovitskiy et al., *An Image is Worth 16x16 Words* (ICLR 2021) · **Course:** 24CS3006 Deep Learning 2026

This notebook runs every experiment in the report on a free Colab **T4** (Runtime ▸ Change runtime type ▸ T4 GPU).
All model code lives in [`vit_lib.py`](vit_lib.py) (written from scratch); this notebook only configures and runs it.

| # | Experiment | Answers |
|---|---|---|
| 1 | ViT vs ResNet-18 **from scratch** on 10 / 25 / 50 / 100 % of CIFAR-10 | Does the missing convolutional inductive bias hurt in the low-data regime? (paper §4.2, Fig. 3–4, in miniature) |
| 2 | **Pre-trained** ViT-B/16 vs ResNet-50, both fine-tuned | Does large-scale pre-training fix it? Compare to the paper's Table 5 |
| 3 | **Ablation:** position embeddings none / 1-D learned / 2-D sin-cos | Which component matters? (paper Appendix D.4) |
| 4 | Attention rollout + confusion matrix + failure cases | Qualitative analysis |

**How to run:** set `PRESET` below → *Run all*. Start with `"dev"` (a couple of minutes, tiny fake data) to check that everything works,
then `"quick"` or `"full"`. Finished runs are cached and interrupted runs resume from the last epoch, so you can safely re-run after a Colab disconnect
(mount Google Drive in the next cell so results survive).

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

try:
    import google.colab                       # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/aditya-kumar04/ViT-CIFAR10.git"
if IN_COLAB:
    if not Path("vit_lib.py").exists():
        if not Path("ViT-CIFAR10").exists():
            subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir("ViT-CIFAR10")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

REPO = Path.cwd()
sys.path.insert(0, str(REPO))

USE_DRIVE = True        # Colab only: keep results + checkpoints on Google Drive so a disconnect loses nothing
OUT = REPO              # where results/ and checkpoints/ live
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/vit_cifar10_runs"); OUT.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import vit_lib as V
import analysis as A
device = V.get_device()
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU (use PRESET='dev' only!)")
print("torch", torch.__version__, "| results dir:", OUT.resolve())

## 1 · Configuration
`PRESET` is the only thing you normally change. *Epoch counts are for the 100 % data runs; see `EQUAL_STEPS` below.*
Rough T4 cost (measure the first epoch's printed time and extrapolate — these are estimates, not measurements):
`quick` ≈ 1.5–2 h in total, `full` ≈ 4–5 h in total.

In [ ]:
PRESET = os.environ.get("VIT_PRESET", "quick")      # "dev" | "quick" | "full"

PRESETS = {
    #           epochs  warmup  mix_prob fractions                 finetune-epochs  fake-data  ft-img  pretrained
    "dev":   dict(epochs=1,   warmup=1,  mix=0.5, fractions=[1.0, 0.5],             ft_epochs=1, fake=128, ft_img=64,  ft_pretrained=False, ft_bs=16),
    "quick": dict(epochs=30,  warmup=3,  mix=0.5, fractions=[1.0, 0.5, 0.25, 0.10], ft_epochs=2, fake=0,   ft_img=224, ft_pretrained=True,  ft_bs=64),
    "full":  dict(epochs=100, warmup=5,  mix=0.8, fractions=[1.0, 0.5, 0.25, 0.10], ft_epochs=3, fake=0,   ft_img=224, ft_pretrained=True,  ft_bs=64),
}
P = PRESETS[PRESET]
if PRESET == "dev":
    OUT = REPO / "dev_out"                         # never mixed with real results
SEEDS      = [42]                                  # sweep seeds; use [42, 43, 44] for mean +- std if time allows
ABL_SEEDS  = [42]                                  # ablation seeds (extend the same way)
EQUAL_STEPS = True                                 # True: smaller subsets train for proportionally MORE epochs (same #updates)
RUN_POOL_ABLATION = False                          # optional extra: CLS token vs mean pooling
DATA_DIR = OUT / "data"
(OUT / "results").mkdir(parents=True, exist_ok=True)
(REPO / "figures").mkdir(exist_ok=True)

VIT = dict(img_size=32, patch_size=4, num_classes=10, embed_dim=192, depth=8, heads=6, mlp_dim=768, drop=0.0)

def vit_cfg(seed, epochs, warmup):
    return V.TrainConfig(epochs=epochs, batch_size=128, optimizer="adamw", lr=1e-3, weight_decay=0.05,
                         warmup_epochs=warmup, label_smoothing=0.1, mix_prob=P["mix"], grad_clip=1.0,
                         amp=True, num_workers=2, seed=seed)

def resnet_cfg(seed, epochs, warmup):
    return V.TrainConfig(epochs=epochs, batch_size=128, optimizer="sgd", lr=0.1, weight_decay=5e-4,
                         warmup_epochs=warmup, label_smoothing=0.1, mix_prob=P["mix"], grad_clip=0.0,
                         amp=True, num_workers=2, seed=seed)

def schedule_for(fraction):
    # epochs / warm-up for a given data fraction
    scale = (1.0 / fraction) if EQUAL_STEPS else 1.0
    return int(round(P["epochs"] * scale)), P["warmup"] * scale

print(f"preset={PRESET}  epochs@100%={P['epochs']}  fractions={P['fractions']}  equal_steps={EQUAL_STEPS}")
for f in P["fractions"]:
    print(f"  fraction {f:.2f}: epochs = {schedule_for(f)[0]}")

## 2 · Data and sanity checks
Fixed 45 000 / 5 000 train/validation split from the 50 000 training images (seed 42); the official 10 000 test images are used **once**, on the checkpoint with the best validation accuracy.
Training augmentation: random crop (reflect-pad 4), horizontal flip, RandAugment(2, 9), then mixup/cutmix and label smoothing (0.1) inside the loop.
Subsets for the data-fraction sweep are **nested** (the 10 % set ⊂ the 25 % set ⊂ …).

In [ ]:
data = V.get_data(DATA_DIR, aug="strong", fake_size=P["fake"])
print(f"train {len(data.train):,} | val {len(data.val):,} | test {len(data.test):,}")
for f in P["fractions"]:
    print(f"  fraction {f:.2f} -> {len(data.train_fraction(f)):,} images")

# --- sanity check 1: the model can overfit a single batch (loss -> ~0).  If this fails, there is a bug.
V.seed_everything(0)
xb, yb = next(iter(V.make_loader(data.val, batch_size=64, num_workers=0)))
xb, yb = xb.to(device), yb.to(device)
m = V.VisionTransformer(**VIT).to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
first = None
for step in range(150):
    loss = F.cross_entropy(m(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
    first = first if first is not None else loss.item()
print(f"overfit-one-batch: loss {first:.3f} -> {loss.item():.4f}")
assert loss.item() < 0.3 * first, "ViT failed to overfit one batch - check the implementation"

# --- sanity check 2: parameter count of the patch embedding = (P*P*C)*D + D
pe = m.patch_embed.proj
print("patch-embed params:", sum(p.numel() for p in pe.parameters()), "= (4*4*3)*192 + 192 =", 4*4*3*192 + 192)
print("ViT total trainable parameters:", V.count_parameters(m))
del m, opt

## 3 · Experiment 1 — ViT vs ResNet-18 from scratch, data-fraction sweep
Both models use the **same** augmentation, mixup/cutmix, label smoothing, schedule shape (linear warm-up + cosine) and number of updates.
Optimisers differ because each is run with its standard recipe: AdamW (lr 1e-3, wd 0.05) for the ViT, Nesterov-SGD (lr 0.1, wd 5e-4) for the ResNet.

In [ ]:
for seed in SEEDS:
    for f in P["fractions"]:
        ep, wu = schedule_for(f)
        sub = data.train_fraction(f)
        V.fit(f"vit-learned_f{f:.2f}_s{seed}", V.VisionTransformer(**VIT), sub, data.val, data.test,
              vit_cfg(seed, ep, wu), OUT, device,
              meta=dict(group="sweep", arch="vit", variant="learned", fraction=f, seed=seed))
        V.fit(f"resnet18_f{f:.2f}_s{seed}", V.build_resnet18_cifar(), sub, data.val, data.test,
              resnet_cfg(seed, ep, wu), OUT, device,
              meta=dict(group="sweep", arch="resnet18", variant="", fraction=f, seed=seed))

In [ ]:
df = A.load_summary(OUT)
A.plot_fraction_sweep(df, REPO / "figures" / "fraction_sweep.png")
s0 = SEEDS[0]
A.plot_curves(OUT, [f"vit-learned_f1.00_s{s0}", f"resnet18_f1.00_s{s0}"], ["ViT", "ResNet-18"],
              REPO / "figures" / "curves_full_data.png")
display(A.agg(df[df.group == "sweep"], ["arch", "fraction"]).round(2))
from IPython.display import Image
Image(str(REPO / "figures" / "fraction_sweep.png"))

## 4 · Experiment 2 — pre-trained ViT-B/16 vs ResNet-50, fine-tuned on CIFAR-10
Pre-trained `timm` backbones (ViT-B/16: `vit_base_patch16_224`, ResNet-50: `resnet50`), new 10-way head, inputs up-sampled 32 → 224 on the GPU.
Fine-tuning recipe follows the paper (SGD + momentum, cosine schedule, gradient clipping at 1). Only a couple of epochs are needed.
If you run out of GPU memory, lower `ft_bs` in the preset (and `lr` roughly proportionally).

In [ ]:
FT_MODELS = ["vit_base_patch16_224", "resnet50"]
for seed in SEEDS:
    for arch in FT_MODELS:
        model, mean, std = V.build_pretrained(arch, img_size=P["ft_img"], pretrained=P["ft_pretrained"])
        d = V.get_data(DATA_DIR, aug="basic", mean=mean, std=std, fake_size=P["fake"])
        cfg = V.TrainConfig(epochs=P["ft_epochs"], batch_size=P["ft_bs"], optimizer="sgd", lr=0.01,
                            weight_decay=0.0, warmup_epochs=0.2, label_smoothing=0.0, mix_prob=0.0,
                            grad_clip=1.0, amp=True, num_workers=2, seed=seed)
        V.fit(f"ft-{arch}_s{seed}", model, d.train, d.val, d.test, cfg, OUT, device,
              meta=dict(group="finetune", arch=arch, variant="pretrained", fraction=1.0, seed=seed))
        del model
        if device.type == "cuda": torch.cuda.empty_cache()

## 5 · Experiment 3 — ablation: position embeddings
Same ViT, same recipe, 100 % of the data. `learned` is the paper's default and is re-used from Experiment 1 (cached).
Following the paper's own ablation: **none** (the model only sees an unordered set of patches), **1-D learned**, **2-D sin-cos** (fixed, no extra parameters).

In [ ]:
variants = ["learned", "none", "sincos2d"] + (["learned-mean"] if RUN_POOL_ABLATION else [])
ep, wu = schedule_for(1.0)
for seed in ABL_SEEDS:
    for v in variants:
        kw = dict(VIT)
        kw["pos_embed"] = v.split("-")[0]
        if v.endswith("mean"): kw["pool"] = "mean"
        V.fit(f"vit-{v}_f1.00_s{seed}", V.VisionTransformer(**kw), data.train_fraction(1.0), data.val, data.test,
              vit_cfg(seed, ep, wu), OUT, device,
              meta=dict(group="ablation", arch="vit", variant=v, fraction=1.0, seed=seed))
df = A.load_summary(OUT)
A.plot_ablation(df, REPO / "figures" / "ablation_pos_embed.png")
Image(str(REPO / "figures" / "ablation_pos_embed.png"))

## 6 · Qualitative analysis — attention rollout, confusion matrix, failure cases
Rollout (Abnar & Zuidema): per layer average the heads, mix in the identity for the residual path (`0.5·A + 0.5·I`), re-normalise rows, multiply layers,
read the CLS row, drop the CLS column, reshape to 8 × 8, up-sample. Rollout is a *diagnostic*, not a causal explanation.

In [ ]:
s0 = SEEDS[0]
best = V.load_best(f"vit-learned_f1.00_s{s0}", V.VisionTransformer(**VIT), OUT, device)
n_ok, n_bad = A.plot_rollout_grid(best, data.test, device, REPO / "figures" / "rollout_grid.png")
print(f"rollout grid: {n_ok} correct, {n_bad} misclassified examples")
info_vit = A.plot_confusion(OUT, f"vit-learned_f1.00_s{s0}", REPO / "figures" / "confusion_vit.png")
info_res = A.plot_confusion(OUT, f"resnet18_f1.00_s{s0}", REPO / "figures" / "confusion_resnet18.png")
print("ViT   most-confused (true, predicted, count):", info_vit["top"][:5])
print("ResNet most-confused (true, predicted, count):", info_res["top"][:5])
print("ViT per-class accuracy:", {k: round(v, 3) for k, v in info_vit["per_class"].items()})
import json
(OUT / "results" / "failure_cases.json").write_text(json.dumps(dict(vit=info_vit, resnet18=info_res), indent=1))
Image(str(REPO / "figures" / "rollout_grid.png"))

## 7 · Collect everything for the report and the repo
Writes `reports/generated/*.tex` (tables + number macros used by the report), `results/summary.csv`, `results/results_table.md` (paste into the README),
and copies the per-run result files into the repo's `results/` folder. Then commit `results/`, `figures/`, `reports/generated/`.

In [ ]:
df = A.build_report_assets(OUT, REPO / "reports" / "generated")
display(df.sort_values(["group", "arch", "variant", "fraction"]).round(2))
if OUT.resolve() != REPO.resolve():                      # Drive / dev: mirror the small result files into the repo
    dst = REPO / ("results" if PRESET != "dev" else "dev_out/results_mirror")
    shutil.copytree(OUT / "results", dst, dirs_exist_ok=True)
print(open(OUT / "results" / "results_table.md").read())

### Next steps (outside the notebook)
1. `git add results figures reports/generated && git commit && git push`
2. `cd reports && pdflatex ViT_CIFAR10_IEEE_Report && bibtex/second pass` (or upload `reports/` to Overleaf) — the report reads the generated tables/macros automatically.
3. **Read the Results/Discussion paragraphs and edit any sentence your numbers do not support** — the text is written around the expected trend, the numbers are whatever you measured.